# 基于MiniLM特征的影评情感识别LRC训练与评估

In [1]:
# 运行准备：按示例代码5.1～5.2读取并划分IMDB数据，复用示例代码5.61和5.63准备MiniLM嵌入特征
import os
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
import sentence_transformers as sentrans

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].tolist()
y = df["label"].tolist()

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at \"{train_file}\" and \"{test_file}\"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].tolist(), df_test["sentence"].tolist()
    y_train, y_test = df_train["label"].tolist(), df_test["label"].tolist()
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    pd.DataFrame({'sentence': sents_train, 'label': y_train}).to_csv(train_file, index=False)
    pd.DataFrame({'sentence': sents_test, 'label': y_test}).to_csv(test_file, index=False)

model = sentrans.SentenceTransformer("./fm/all-MiniLM-L6-v2")
x_train = model.encode(sents_train)
x_test = model.encode(sents_test)

Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
clf = LogisticRegression()
clf.fit(x_train, y_train)
pred = clf.predict(x_test)
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.914     0.914     0.914       105
           1      0.905     0.905     0.905        95

    accuracy                          0.910       200
   macro avg      0.910     0.910     0.910       200
weighted avg      0.910     0.910     0.910       200

